In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['OPENAI_API_KEY']:
    print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


In [4]:
from langchain_openai import ChatOpenAI
# from langchain_core import PromptTemplate

In [ ]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

In [ ]:
response = llm.invoke("What is AI? Tell me in one sentence.")
response.content

'AI is the field of computer science focused on creating systems that can perform tasks that normally require human intelligence, such as learning, reasoning, perception, and language understanding.'

# RAG Implementation with data

#### STEP 1: Preparing document for text 

In [17]:
from langchain_core.documents import Document

In [18]:
my_text = """Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: A lot of cutting edge AI has filtered into general applications, often without being called AI because once something becomes useful enough and common enough it's not labeled AI anymore.[2][3]

Various subfields of AI research are centered around particular goals and the use of particular tools. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.[a] To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[4] Some companies, such as OpenAI, Google DeepMind and Meta,[5] aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.

Artificial intelligence was founded as an academic discipline in 1956,[6] and the field went through multiple cycles of optimism throughout its history,[7][8] followed by periods of disappointment and loss of funding, known as AI winters.[9][10] Funding and interest vastly increased after 2012 when graphics processing units started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[11] This growth accelerated further after 2017 with the transformer architecture.[12] In the 2020s, an ongoing period of rapid progress in advanced generative AI became known as the AI boom. Generative AI's ability to create and modify content has led to several unintended consequences and harms. Ethical concerns have been raised about AI's long-term effects and potential existential risks, prompting discussions about regulatory policies to ensure the safety and benefits of the technology."""

docs = [Document(page_content=my_text, metadata={"source": "ABC", "documentID":"Doc1"})]

In [19]:
print(docs)

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content="Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: A lot of cu

#### STEP 2: Splitting the documents into Chunks

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess a

#### STEP 3: Creating Embeddings on Chunks

In [34]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [35]:
embedding_model.embed_query("What is AI?")

[-0.018027925863862038,
 0.012509824708104134,
 -0.007017239462584257,
 0.008082584477961063,
 0.016267236322164536,
 -0.04059791937470436,
 -0.009932584129273891,
 -0.013141375966370106,
 -0.041414473205804825,
 0.03033999167382717,
 -0.00029424560489133,
 -0.0472579188644886,
 -0.02242964878678322,
 -0.055729642510414124,
 -0.011750686913728714,
 0.0036521542351692915,
 -0.023756545037031174,
 -0.01793861575424671,
 0.026869649067521095,
 -0.04577792063355446,
 0.004293274600058794,
 0.044910334050655365,
 -0.011916548945009708,
 0.002500689122825861,
 0.020592408254742622,
 -0.03975585103034973,
 0.03597930073738098,
 0.0045133610256016254,
 -0.026486890390515327,
 -0.014276892878115177,
 0.034193094819784164,
 -0.027890337631106377,
 0.004519740119576454,
 -0.01652240939438343,
 0.013205168768763542,
 0.02582344226539135,
 -0.015603789128363132,
 0.01130413543432951,
 0.0027606459334492683,
 -0.021523786708712578,
 0.014595858752727509,
 0.005719050299376249,
 0.02288896031677723,


#### STEP 4: Create and Store Embeddings in Vector Store

In [27]:
from langchain_community.vectorstores import Chroma 

vectorstore = Chroma.from_documents(chunks, embedding_model)

In [28]:
vectorstore.similarity_search("What is AI?", k=2)

[Document(metadata={'documentID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'documentID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess a

In [29]:
context = vectorstore.similarity_search("What is AI?", k=2)

In [33]:
response = llm.invoke(f"What is AI? You can answer using the following context: {context}")
print(response.content)

AI, or artificial intelligence, is the capability of computer systems to perform tasks that are typically associated with human intelligence. This includes learning, reasoning, problem-solving, perception, and decision-making. It’s a field of computer science that develops methods and software enabling machines to perceive their environment and use learning and intelligence to take actions aimed at achieving defined goals.

Common examples:
- Web search engines (e.g., Google Search)
- Recommendation systems (YouTube, Amazon, Netflix)
- Virtual assistants (Google Assistant, Siri, Alexa)
- Autonomous vehicles (Waymo)
- Generative and creative tools (language models, AI art)
- Strategy game play and analysis (chess, Go)

Note: many AI applications aren’t labeled as AI to the public; AI has increasingly filtered into everyday technology.
